In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
import time
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-optimization-preparation").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 15:36:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Spark Optimization

Even correct Spark code can be slow. The main levers are:

| Technique | When to use |
|-----------|-------------|
| **Data formats** | Always prefer Parquet/ORC over CSV for analytics |
| **Caching / persist** | When the same DataFrame is used multiple times |
| **Partitioning** | When queries filter on a column (write-time optimization) |
| **Broadcast joins** | When one side of a join fits in worker memory |

In this notebook we measure the effect of each technique with timing comparisons.

References:
- https://spark.apache.org/docs/latest/sql-performance-tuning.html
- https://spark.apache.org/docs/latest/rdd-programming-guide.html#rdd-persistence

# Example 1: Data Formats -- CSV vs Parquet

**Parquet** is a columnar storage format that:
- Stores data column-by-column (ideal for analytical queries that touch only a few columns)
- Embeds schema information -- no inference needed on read
- Compresses efficiently, especially numeric data
- Supports **predicate pushdown** -- filters applied before reading from disk

We compare CSV and Parquet using Cambridge, MA public salary data.

In [2]:
def timed(fn):
    t0 = time.time()
    result = fn()
    return result, time.time() - t0

# Read from CSV
_, t_csv = timed(lambda: (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("data/Cambridge_Budget_Salaries.csv")
    .select(F.avg(F.col("Total Salary"))).first()))
print(f"CSV read + avg:     {t_csv:.3f}s")

CSV read + avg:     1.670s


In [3]:
# Read from Parquet (same data, different format)
_, t_parquet = timed(lambda: (spark.read
    .parquet("data/Cambridge_Budget_Salaries.parquet")
    .select(F.avg(F.col("Total_Salary"))).first()))
print(f"Parquet read + avg: {t_parquet:.3f}s")
print(f"Speedup: {t_csv/max(t_parquet,0.001):.1f}×  (effect is pronounced on large files)")

Parquet read + avg: 0.480s
Speedup: 3.5×  (effect is pronounced on large files)


In [4]:
# Aggregate by department -- works best with Parquet (column pruning)
df_parquet = spark.read.parquet("data/Cambridge_Budget_Salaries.parquet")
df_parquet.groupBy("Service").avg("Total_Salary").orderBy("avg(Total_Salary)", ascending=False).show(10)

+--------------------+-----------------+
|             Service|avg(Total_Salary)|
+--------------------+-----------------+
|  General Government|84782.28296703297|
|       Public Safety|75823.88040638607|
|Community Mainten...|70001.38853948288|
|Human Resources a...|52131.12394366197|
+--------------------+-----------------+



In [5]:
# Write CSV data to Parquet -- rename columns to remove spaces (Parquet restriction)
import shutil, os
out = "tmp/Cambridge_Salaries_parquet"
if os.path.exists(out): shutil.rmtree(out)

df_csv = spark.read.option("header","true").option("inferSchema","true").csv("data/Cambridge_Budget_Salaries.csv")
renamed = df_csv
for col_name in df_csv.columns:
    renamed = renamed.withColumnRenamed(col_name, col_name.replace(' ', '_'))
renamed.write.parquet(out)
print(f"Written Parquet to {out}")

Written Parquet to tmp/Cambridge_Salaries_parquet


# Example 2: Caching and Persistence

When you reuse a DataFrame in multiple actions, Spark **recomputes it from scratch each time** because transformations are lazy. `df.cache()` (= `df.persist(StorageLevel.MEMORY_AND_DISK)`) stores the result after the first action.

Use `df.unpersist()` to release memory when done.

In [6]:
df_big = spark.range(5_000_000).toDF("id").withColumn("square", F.col("id") * F.col("id"))

# Without caching -- recomputed each time
_, t1 = timed(lambda: df_big.count())
_, t2 = timed(lambda: df_big.count())
print(f"Without cache: 1st={t1:.3f}s  2nd={t2:.3f}s  (both recomputed)")

Without cache: 1st=0.161s  2nd=0.049s  (both recomputed)


In [7]:
df_big.cache()   # marks the DataFrame for caching -- actually cached on first action

_, t1 = timed(lambda: df_big.count())   # compute + cache
_, t2 = timed(lambda: df_big.count())   # served from cache
_, t3 = timed(lambda: df_big.count())   # served from cache
print(f"With cache:    1st={t1:.3f}s  2nd={t2:.3f}s  3rd={t3:.3f}s")

df_big.unpersist()
print("Unpersisted -- memory freed")

With cache:    1st=0.602s  2nd=0.097s  3rd=0.056s
Unpersisted -- memory freed


# Example 3: Partitioning Data

**Write-time partitioning** organizes data on disk by column value.  
Subsequent reads that filter on that column only open the relevant partition directories (**partition pruning** -- skips irrelevant data entirely).

This is critical for large datasets partitioned by date, state, or category.

In [9]:
# Fannie Mae loan data -- partition by state (_c30)
data = (spark.read
    .option("header", "false")
    .option("delimiter", "|")
    .csv("data/sf-loan-performance-data-sample.csv"))

out = "tmp/fannie_mae_partitioned"
if os.path.exists(out): shutil.rmtree(out)
data.write.partitionBy("_c30").option("delimiter", "|").csv(out)
print("Written partitioned data")

26/06/12 15:38:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Written partitioned data


In [10]:
partitions = os.listdir(out)
print(f"Partition directories: {len(partitions)}")
print("Sample:", sorted(partitions)[:5])

Partition directories: 10
Sample: ['._SUCCESS.crc', '_SUCCESS', '_c30=CA', '_c30=FL', '_c30=MN']


In [11]:
# Reading with filter -- Spark only opens the _c30=FL directory
_, t = timed(lambda: spark.read.option("delimiter","|").csv(f"{out}/_c30=FL").count())
print(f"FL loans -- read from single partition in {t:.3f}s")

FL loans -- read from single partition in 0.139s


In [22]:
_, t = timed(lambda: spark.read
        .option("header", "false")
    .option("delimiter", "|")
    .csv("data/sf-loan-performance-data-sample.csv")
    .where(F.col('_c30')=='FL').count())
t

0.12915897369384766

In [23]:
_, t = timed(lambda: spark.read.csv("tmp/fannie_mae_partitioned").where(F.col('_c30')=='FL').count())
t

0.1282801628112793

# Example 4: Optimizing Joins -- Sort Merge vs Broadcast

Spark chooses a join strategy automatically:

| Strategy | Mechanism | When used |
|----------|-----------|-----------|
| **Sort Merge Join** | Both sides sorted and merged -- requires a shuffle | Large tables |
| **Broadcast Join** | Small table sent to every worker -- no shuffle | One side < `autoBroadcastJoinThreshold` (default 10 MB) |

Use `F.broadcast(small_df)` to force a broadcast join, or configure `spark.sql.autoBroadcastJoinThreshold`.

In [24]:
# Disable auto-broadcast to force Sort Merge Join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 0)

trips = (spark.read.option("header","true").option("inferSchema","false")
         .csv("data/green_tripdata_2020-04.csv.gz"))
zones = (spark.read.option("header","true").csv("data/taxi_zone_lookup.csv"))
print(f"Trips: {trips.count():,}  |  Zones: {zones.count()}")

Trips: 35,612  |  Zones: 265


In [25]:
jfk_count, t_merge = timed(lambda:
    trips.join(zones, trips["DOLocationID"] == zones["LocationID"])
         .filter(F.col("Zone") == "JFK Airport").count())
print(f"Sort Merge Join -- JFK trips: {jfk_count}, time: {t_merge:.3f}s")

Sort Merge Join -- JFK trips: 24, time: 0.339s


In [26]:
# Re-enable auto-broadcast; force broadcast explicitly with F.broadcast()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

jfk_count2, t_bcast = timed(lambda:
    trips.join(F.broadcast(zones), trips["DOLocationID"] == zones["LocationID"])
         .filter(F.col("Zone") == "JFK Airport").count())
print(f"Broadcast Join   -- JFK trips: {jfk_count2}, time: {t_bcast:.3f}s")
print(f"Speedup: {t_merge/max(t_bcast,0.001):.1f}×")

Broadcast Join   -- JFK trips: 24, time: 0.191s
Speedup: 1.8×


# Shutdown Spark when done

In [27]:
spark.stop()